# Assignment 8: Text Processing with NLTK

## Q1. Tokenization, stopword removal, and word frequency

This paragraph is about technology:

> I enjoy exploring how technology can make everyday learning more engaging. A well-designed app can turn a difficult idea into a small, interactive experiment. I often use online tutorials to revisit topics and practice at my own pace. Still, thoughtful design matters because too many notifications can distract learners. The best tools support curiosity while leaving room for focused thinking.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize

required_resources = ("punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4")
failed_resources = [
    resource for resource in required_resources
    if not nltk.download(resource, quiet=True)
]
if failed_resources:
    raise RuntimeError(f"Could not download NLTK resources: {failed_resources}")

paragraph = (
    "I enjoy exploring how technology can make everyday learning more engaging. "
    "A well-designed app can turn a difficult idea into a small, interactive experiment. "
    "I often use online tutorials to revisit topics and practice at my own pace. "
    "Still, thoughtful design matters because too many notifications can distract learners. "
    "The best tools support curiosity while leaving room for focused thinking."
)

lower_text = paragraph.lower()
clean_text = re.sub(r"[^\w\s]", "", lower_text)
print("Lowercase text with punctuation removed:")
print(clean_text)

### Q1.1–Q1.5: Sentence/word tokens, comparison, stopwords, and frequency

`split()` separates at whitespace, so punctuation remains attached to words. NLTK's `word_tokenize()` recognizes punctuation as separate tokens. Both comparisons below use the lowercased original text; the cleaned tokens are then used for stopword removal and frequency counts.

In [ ]:
sentences = sent_tokenize(paragraph)
print("Sentences:")
for sentence in sentences:
    print("-", sentence)

split_tokens = lower_text.split()
nltk_tokens_with_punctuation = word_tokenize(lower_text)
cleaned_tokens = word_tokenize(clean_text)

print("\nPython split() tokens:")
print(split_tokens)
print("\nNLTK word_tokenize() tokens:")
print(nltk_tokens_with_punctuation)
print("\nNLTK tokens after punctuation removal:")
print(cleaned_tokens)

stop_words = set(stopwords.words("english"))
filtered_tokens = [word for word in cleaned_tokens if word not in stop_words]
print("\nTokens after stopword removal:")
print(filtered_tokens)

frequency_distribution = FreqDist(filtered_tokens)
print("\nWord frequency distribution (excluding stopwords):")
for word, count in frequency_distribution.most_common():
    print(f"{word}: {count}")

## Q2. Alphabetic words, stemming, and lemmatization

The regular expression below extracts alphabetic sequences only. Stopwords are removed before applying Porter stemming and WordNet lemmatization.

In [ ]:
alphabetic_words = re.findall(r"\b[a-zA-Z]+\b", paragraph.lower())
q2_tokens = [word for word in alphabetic_words if word not in stop_words]

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stemmed_words = [stemmer.stem(word) for word in q2_tokens]
lemmatized_words = [lemmatizer.lemmatize(word) for word in q2_tokens]

print("Alphabetic words:")
print(alphabetic_words)
print("\nAlphabetic words after stopword removal:")
print(q2_tokens)
print("\nPorter-stemmed words:")
print(stemmed_words)
print("\nWordNet-lemmatized words (default noun part of speech):")
print(lemmatized_words)

print("\nInput → stemmed → lemmatized:")
for original, stem, lemma in zip(q2_tokens, stemmed_words, lemmatized_words):
    print(f"{original} → {stem} → {lemma}")

### Comparison and when to use each

Porter stemming uses heuristic rules to chop word endings, so its results may not be valid words (for example, *notifications* can become *notif*). Lemmatization uses vocabulary and a part-of-speech assumption to return a dictionary form (as a noun, *notifications* becomes *notification*). Stemming is usually preferable when speed and broad matching are priorities, such as a search index; lemmatization is preferable when readable, linguistically meaningful forms matter, such as in text analysis or user-facing output. This example uses WordNet's default noun part of speech; supplying the correct part of speech can improve lemmatization.

## Q3. Bag of Words and TF-IDF

Three short technology-related texts are vectorized. CountVectorizer records term counts, while TF-IDF gives more weight to terms that are informative within one text and less common across all three.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

q3_texts = [
    "Solar panels convert sunlight into clean electricity for homes.",
    "A bicycle sharing app helps commuters find nearby bikes and plan routes.",
    "Smart speakers answer questions and control lights with voice commands."
]

count_vectorizer = CountVectorizer(stop_words="english")
count_matrix = count_vectorizer.fit_transform(q3_texts)
count_terms = count_vectorizer.get_feature_names_out()

print("Bag of Words counts (term: count for each text):")
for index, text in enumerate(q3_texts):
    counts = {
        term: int(count_matrix[index, column])
        for column, term in enumerate(count_terms)
        if count_matrix[index, column] > 0
    }
    print(f"Text {index + 1}: {counts}")

tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(q3_texts)
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

print("\nTF-IDF scores and top 3 keywords per text:")
for index, text in enumerate(q3_texts):
    row = tfidf_matrix[index].toarray().ravel()
    nonzero_columns = row.nonzero()[0]
    top_columns = sorted(nonzero_columns, key=lambda column: row[column], reverse=True)[:3]
    scores = {tfidf_terms[column]: round(float(row[column]), 3) for column in nonzero_columns}
    keywords = [(tfidf_terms[column], round(float(row[column]), 3)) for column in top_columns]
    print(f"\nText {index + 1}: {text}")
    print("All TF-IDF scores:", scores)
    print("Top 3 keywords:", keywords)
    print("Interpretation: these highest-scoring terms summarize the distinctive subject of this text.")

## Q4. Similarity between two technologies

Each description has four lines. They are normalized to lowercase alphabetic tokens before comparing their vocabularies and TF-IDF vectors.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

ai_text = """Artificial intelligence learns patterns from examples.
It can recognize images, summarize language, and answer questions.
AI systems help people automate repetitive decisions.
Their results depend on data quality and careful oversight."""

blockchain_text = """Blockchain stores transactions in linked digital blocks.
Many computers share and verify the same record.
This technology supports transparent exchanges without one central owner.
Its security depends on consensus and careful network design."""

def preprocess(text):
    return re.findall(r"\b[a-z]+\b", text.lower())

ai_tokens = preprocess(ai_text)
blockchain_tokens = preprocess(blockchain_text)
ai_set = set(ai_tokens)
blockchain_set = set(blockchain_tokens)
union = ai_set | blockchain_set
jaccard = len(ai_set & blockchain_set) / len(union) if union else 0.0

q4_vectorizer = TfidfVectorizer(tokenizer=preprocess, token_pattern=None, lowercase=False)
q4_tfidf = q4_vectorizer.fit_transform([ai_text, blockchain_text])
cosine = float(cosine_similarity(q4_tfidf[0:1], q4_tfidf[1:2])[0, 0])

print("AI tokens:", ai_tokens)
print("Blockchain tokens:", blockchain_tokens)
print("Shared vocabulary:", sorted(ai_set & blockchain_set))
print(f"Jaccard similarity: {jaccard:.3f}")
print(f"TF-IDF cosine similarity: {cosine:.3f}")

**Analysis:** Jaccard similarity compares only whether a word appears, treating every shared word equally. TF-IDF cosine similarity also compares the vectors' term weights, so it accounts for how informative the shared terms are and for differences in document length. Here, cosine similarity generally gives the more nuanced comparison, while Jaccard is easier to explain as simple vocabulary overlap; neither metric understands meaning or synonyms.

## Q5. Review sentiment and word cloud

TextBlob polarity ranges from -1 (negative) to +1 (positive); its subjectivity score ranges from 0 (objective) to 1 (subjective). The thresholds below label scores above 0.1 positive, below -0.1 negative, and the rest neutral.

In [ ]:
from textblob import TextBlob
from wordcloud import WordCloud
import matplotlib.pyplot as plt

reviews = [
    "I love these comfortable headphones. The sound is clear, the battery lasts all day, and the controls are easy to use.",
    "This desk lamp is excellent. It is bright, stylish, and simple to adjust while I work.",
    "The delivery was late and the headphones stopped working. The poor sound and unhelpful support made this a frustrating purchase.",
    "The package arrived on Tuesday. It contains a blue notebook with a paper cover."
]

positive_reviews = []
print("Review sentiment results:")
for review in reviews:
    sentiment = TextBlob(review).sentiment
    if sentiment.polarity > 0.1:
        label = "Positive"
        positive_reviews.append(review)
    elif sentiment.polarity < -0.1:
        label = "Negative"
    else:
        label = "Neutral"
    print(f"\nReview: {review}")
    print(f"Polarity: {sentiment.polarity:.3f} | Subjectivity: {sentiment.subjectivity:.3f} | Class: {label}")

if not positive_reviews:
    raise ValueError("No positive reviews were classified, so a word cloud cannot be generated.")

positive_text = " ".join(positive_reviews)
wordcloud = WordCloud(width=800, height=400, background_color="white").generate(positive_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud: Positive Reviews")
plt.show()

## Q6. Next-word generation with a small LSTM

This roughly 100-word paragraph is tokenized with Keras' `Tokenizer`. Prefixes of increasing length become input sequences, with the next word as each target. The small model is trained on this paragraph only, so generated text demonstrates the method rather than producing high-quality language. TensorFlow/Keras is required for this section.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

training_text = (
    "Technology can help people learn when it is designed with care. "
    "A curious learner asks questions, tests ideas, and learns from mistakes. "
    "Digital tools can offer lessons, examples, and practice at any time. "
    "A useful learning app explains difficult ideas in simple steps. "
    "It gives learners space to explore, repeat a lesson, and track progress. "
    "Teachers can use technology to share resources and give helpful feedback. "
    "However, a screen cannot replace patience, conversation, or imagination. "
    "Good learning combines clear goals with active practice and reflection. "
    "When people stay curious, each new challenge becomes a chance to grow. "
    "The best tools support thoughtful learning and help ideas connect."
)

tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
word_sequences = tokenizer.texts_to_sequences([training_text])[0]
input_sequences = [word_sequences[:index + 1] for index in range(1, len(word_sequences))]
max_sequence_len = max(len(sequence) for sequence in input_sequences)
padded_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding="pre")
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]
vocab_size = len(tokenizer.word_index) + 1

print(f"Training words: {len(word_sequences)} | Vocabulary size: {vocab_size - 1}")
print(f"Training sequences: {X.shape[0]} | Input sequence length: {X.shape[1]}")

tf.keras.utils.set_random_seed(42)
model = Sequential([
    Embedding(vocab_size, 24, input_length=max_sequence_len - 1),
    LSTM(48),
    Dense(vocab_size, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(X, y, epochs=80, verbose=0)
print("LSTM training complete.")

def generate_line(seed, words_to_generate=8):
    generated = seed.lower()
    for _ in range(words_to_generate):
        sequence = tokenizer.texts_to_sequences([generated])[0]
        sequence = pad_sequences([sequence], maxlen=max_sequence_len - 1, padding="pre")
        probabilities = model.predict(sequence, verbose=0)[0]
        next_id = int(np.argmax(probabilities))
        next_word = tokenizer.index_word.get(next_id)
        if next_word is None:
            break
        generated += " " + next_word
    return generated

print("\nGenerated lines:")
for seed in ("technology", "learning", "curious"):
    print("-", generate_line(seed))